# ScamRoad — Autonomous Kaggle Cloud GPU Training & Optimization Pipeline

**Repository**: [github.com/Priya-Ranjan-0201/potholes](https://github.com/Priya-Ranjan-0201/potholes)  
**Compute Target**: Kaggle GPU (Tesla T4 x 2 or P100 with CUDA)  
**Persistent Hub**: `/kaggle/working/artifacts`  
**Architectures Compared**:
1. **YOLO11n (Nano)** — Lightweight mobile edge detector (<10ms latency)
2. **YOLO11s (Small)** — Higher capacity detector for fine road cracks
3. **YOLO11n-seg (Segmentation)** — Instance segmentation for damage surface area
4. **RT-DETR-l (Transformer)** — End-to-end real-time transformer detector
5. **YOLOv8n (Baseline)** — Reference edge mobile baseline
**Taxonomy**: `pothole`, `crack_longitudinal`, `crack_transverse`, `crack_alligator`


## Step 1: System & Dual GPU Resource Audit
Audit available CUDA devices, VRAM capacity, and environment details.

In [ ]:
# 1. GPU & System Resource Audit
!nvidia-smi

import psutil, shutil, torch
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/kaggle/working')

print('=' * 65)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU [{i}] Device   : {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)')
    print(f'CUDA Version     : {torch.version.cuda}')
print(f'System RAM       : {ram.total / (1024**3):.2f} GB (Available: {ram.available / (1024**3):.2f} GB)')
print(f'Working Disk Free: {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB')
print('=' * 65)


## Step 2: Clone Repository & Install Cloud Dependencies

In [ ]:
# 2. Clone repo and install dependencies
import os, sys
from pathlib import Path

repo_dir = Path('/kaggle/working/potholes')
if not repo_dir.exists():
    !git clone https://github.com/Priya-Ranjan-0201/potholes.git /kaggle/working/potholes
else:
    !cd /kaggle/working/potholes && git pull origin main

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
os.chdir(str(repo_dir))

!pip install -q ultralytics onnx onnxruntime-gpu pyyaml albumentations opencv-python
print('[SUCCESS] Workspace setup complete at:', repo_dir.resolve())


## Step 3: Dataset Ingestion & Standardized Splitting
Extracts attached Kaggle datasets or downloads them, converts annotations, and creates sequence splits.

In [ ]:
# 3. Ingest Datasets
import shutil
from pathlib import Path

raw_dir = Path('/kaggle/working/data/raw/potholes')
raw_dir.mkdir(parents=True, exist_ok=True)

# Check if dataset is attached in /kaggle/input
kaggle_input = Path('/kaggle/input/pothole-detection')
if kaggle_input.exists():
    print('[INFO] Using attached Kaggle input dataset: /kaggle/input/pothole-detection')
    src_ann = kaggle_input / 'annotations'
    src_img = kaggle_input / 'images'
else:
    print('[INFO] Pulling dataset via Kaggle CLI...')
    !kaggle datasets download -d larxel/pothole-detection -p /kaggle/working/data/raw/potholes --unzip
    src_ann = raw_dir / 'annotations'
    src_img = raw_dir / 'images'

# Convert VOC to YOLO
!python -m ml.data.convert --format voc --xml-dir "{src_ann}" --image-dir "{src_img}" --output-dir /kaggle/working/data/processed/potholes/labels

# Stage images into processed folder
proc_img = Path('/kaggle/working/data/processed/potholes/images')
proc_img.mkdir(parents=True, exist_ok=True)
for img in src_img.glob('*.*'):
    if img.is_file():
        shutil.copy2(img, proc_img / img.name)

# Audit dataset
!python -m ml.data.audit --image-dir /kaggle/working/data/processed/potholes/images --label-dir /kaggle/working/data/processed/potholes/labels

# Split into sequence-aware 70/15/15 partitions
!python -m ml.data.split --image-dir /kaggle/working/data/processed/potholes/images --label-dir /kaggle/working/data/processed/potholes/labels --output-dir /kaggle/working/data/splits --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

# Generate the 7 canonical manifests
!python -m ml.data.clean_and_unify --data-dir /kaggle/working/data/splits --output-dir /kaggle/working/artifacts/dataset_manifests
print('[SUCCESS] Dataset ready for training.')


## Step 4: Model 1 — Train YOLO11n (Nano Mobile Edge Detector)

In [ ]:
# 4. Train YOLO11n
from ultralytics import YOLO
from pathlib import Path

runs_dir = Path('/kaggle/working/artifacts/runs')
dataset_cfg = Path('/kaggle/working/data/splits/dataset.yaml')

base_weight = 'yolo11n.pt'
try:
    model_11n = YOLO(base_weight)
except Exception:
    base_weight = 'yolov8n.pt'
    model_11n = YOLO(base_weight)

print(f'[INFO] Training {base_weight} on Kaggle GPU...')
results_11n = model_11n.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(runs_dir),
    name='model_yolo11n',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11n training completed.')


## Step 5: Model 2 — Train YOLO11s (Small Road Defect Detector)

In [ ]:
# 5. Train YOLO11s
base_s = 'yolo11s.pt'
try:
    model_11s = YOLO(base_s)
except Exception:
    base_s = 'yolov8s.pt'
    model_11s = YOLO(base_s)

print(f'[INFO] Training {base_s} on Kaggle GPU...')
results_11s = model_11s.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(runs_dir),
    name='model_yolo11s',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11s training completed.')


## Step 6: Model 3 — Train YOLO11n-seg (Instance Segmentation)

In [ ]:
# 6. Train YOLO11n-seg
base_seg = 'yolo11n-seg.pt'
try:
    model_seg = YOLO(base_seg)
except Exception:
    base_seg = 'yolov8n-seg.pt'
    model_seg = YOLO(base_seg)

print(f'[INFO] Training {base_seg} instance segmentation on Kaggle GPU...')
results_seg = model_seg.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(runs_dir),
    name='model_yolo11n_seg',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11n-seg training completed.')


## Step 7: Model 4 — Train RT-DETR-l (Real-Time Transformer)

In [ ]:
# 7. Train RT-DETR-l
model_rtdetr = YOLO('rtdetr-l.pt')
results_rtdetr = model_rtdetr.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project=str(runs_dir),
    name='model_rtdetr_l',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] RT-DETR-l training completed.')


## Step 8: Multi-Model Evaluation & Pareto Comparison Harness

In [ ]:
# 8. Multi-Model Benchmark Harness
comp_dir = Path('/kaggle/working/artifacts/comparisons')

!python -m ml.evaluation.compare_models \
    --data /kaggle/working/data/splits/dataset.yaml \
    --models \
        YOLO11n=/kaggle/working/artifacts/runs/model_yolo11n/weights/best.pt \
        YOLO11s=/kaggle/working/artifacts/runs/model_yolo11s/weights/best.pt \
        YOLO11n-seg=/kaggle/working/artifacts/runs/model_yolo11n_seg/weights/best.pt \
        RT-DETR-l=/kaggle/working/artifacts/runs/model_rtdetr_l/weights/best.pt \
    --out-dir /kaggle/working/artifacts/comparisons \
    --device 0

from IPython.display import Markdown, display
report_file = comp_dir / 'model_comparison.md'
if report_file.exists():
    display(Markdown(report_file.read_text()))


## Step 9: ONNX Dynamic INT8 Quantization

In [ ]:
# 9. Dynamic INT8 Quantization
import shutil
from pathlib import Path
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType

exports_dir = Path('/kaggle/working/artifacts/exports')
exports_dir.mkdir(parents=True, exist_ok=True)

candidate_models = [
    runs_dir / 'model_yolo11n' / 'weights' / 'best.pt',
    runs_dir / 'model_yolo11s' / 'weights' / 'best.pt',
    runs_dir / 'model_yolo11n_seg' / 'weights' / 'best.pt',
    runs_dir / 'model_rtdetr_l' / 'weights' / 'best.pt',
]
best_pt = next((p for p in candidate_models if p.exists()), None)

if best_pt:
    print(f'[INFO] Quantizing top model: {best_pt}')
    model = YOLO(str(best_pt))
    onnx_file = model.export(format='onnx', imgsz=640, simplify=True)
    fp32_dest = exports_dir / 'scamroad_best_fp32.onnx'
    shutil.copy2(onnx_file, fp32_dest)

    int8_dest = exports_dir / 'scamroad_best_int8.onnx'
    quantize_dynamic(
        model_input=str(fp32_dest),
        model_output=str(int8_dest),
        weight_type=QuantType.QUInt8
    )
    print(f'[SUCCESS] FP32 ONNX : {fp32_dest} ({fp32_dest.stat().st_size / (1024**2):.2f} MB)')
    print(f'[SUCCESS] INT8 ONNX : {int8_dest} ({int8_dest.stat().st_size / (1024**2):.2f} MB)')
else:
    print('[WARNING] No best.pt found to quantize.')


## Step 10: Multi-Frame Defect Tracking & Video HUD Verification

In [ ]:
# 10. Infer Media and Test Video HUD Tracking
selected_weight = next((str(p) for p in candidate_models if p.exists()), 'yolo11n.pt')

!python scripts/infer_media.py \
    --source /kaggle/working/data/splits/images/val \
    --weights "{selected_weight}" \
    --output-dir /kaggle/working/artifacts/inference_demo \
    --conf 0.25

print('[SUCCESS] Media inference demo complete.')


## Step 11: Final Deliverables & Artifacts Inventory

In [ ]:
# 11. Final Artifacts Listing
!ls -lhR /kaggle/working/artifacts
print('\n[COMPLETE] ScamRoad Kaggle GPU Pipeline execution finished successfully.')
